<a href="https://colab.research.google.com/github/JordanTerwilliger/Intro-to-Deep-Learning/blob/main/HW5/HW5_ViT_Transformer_Q2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

import torch as torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torchsummary import summary
!pip install torchinfo
from torchinfo import summary
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights
from tqdm import tqdm
import matplotlib.pyplot as plt

import time
torch.manual_seed(1)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [2]:
# Hyperparameters
image_size = 224
num_classes = 100
num_epochs = 5
learning_rate = 2e-5
batch_size = 32

In [3]:
# Load model directly
from transformers import SwinForImageClassification, AutoImageProcessor, SwinConfig

# Data preparation with proper preprocessing for Swin
processor = AutoImageProcessor.from_pretrained("microsoft/swin-tiny-patch4-window7-224")

transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std)
])


train_data = torchvision.datasets.CIFAR100(
    root='./data', train=True, transform=transform, download=True)
test_data = torchvision.datasets.CIFAR100(
    root='./data', train=False, transform=transform, download=True)

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle = True, num_workers=2, pin_memory=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle = True, num_workers=2, pin_memory=True)


preprocessor_config.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

In [4]:
# Patch embedding (same idea as ViT, but we keep spatial layout instead of flattening to a sequence immediately)
class PatchEmbedding(nn.Module):
    def __init__(self, image_size, patch_size, in_channels=3, embed_dim=96):
        super().__init__()
        self.image_size = image_size
        self.patch_size = patch_size
        self.grid_size = image_size // patch_size
        self.num_patches = self.grid_size ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim,
                               kernel_size=patch_size, stride=patch_size)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.proj(x)                      # [B, embed_dim, H', W']
        x = x.flatten(2).transpose(1, 2)       # [B, num_patches, embed_dim]
        x = self.norm(x)
        return x


def window_partition(x, window_size, H, W):
    # x: [B, H*W, C] -> windows: [num_windows*B, window_size*window_size, C]
    B, N, C = x.shape
    x = x.view(B, H, W, C)
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous()
    windows = windows.view(-1, window_size * window_size, C)
    return windows


def window_reverse(windows, window_size, H, W):
    # windows: [num_windows*B, window_size*window_size, C] -> x: [B, H*W, C]
    C = windows.shape[-1]
    B = int(windows.shape[0] / (H * W / window_size / window_size))
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, C)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous()
    x = x.view(B, H, W, C)
    x = x.view(B, H * W, C)
    return x


class WindowAttention(nn.Module):
    """Multi-head self-attention restricted to a local window, with relative position bias."""
    def __init__(self, embed_dim, window_size, num_heads, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.window_size = window_size
        self.num_heads = num_heads
        head_dim = embed_dim // num_heads
        self.scale = head_dim ** -0.5

        # Relative position bias table: (2*Wh-1) * (2*Ww-1) entries per head
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size - 1) * (2 * window_size - 1), num_heads)
        )

        coords_h = torch.arange(window_size)
        coords_w = torch.arange(window_size)
        coords = torch.stack(torch.meshgrid(coords_h, coords_w, indexing="ij"))  # [2, Wh, Ww]
        coords_flatten = torch.flatten(coords, 1)  # [2, Wh*Ww]
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]  # [2, N, N]
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()  # [N, N, 2]
        relative_coords[:, :, 0] += window_size - 1
        relative_coords[:, :, 1] += window_size - 1
        relative_coords[:, :, 0] *= 2 * window_size - 1
        relative_position_index = relative_coords.sum(-1)  # [N, N]
        self.register_buffer("relative_position_index", relative_position_index)

        nn.init.trunc_normal_(self.relative_position_bias_table, std=0.02)

        self.qkv = nn.Linear(embed_dim, embed_dim * 3, bias=True)
        self.attn_drop = nn.Dropout(dropout)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.proj_drop = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # x: [num_windows*B, N, C], N = window_size*window_size
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # [3, B_, heads, N, head_dim]
        q, k, v = qkv[0], qkv[1], qkv[2]

        q = q * self.scale
        attn = q @ k.transpose(-2, -1)  # [B_, heads, N, N]

        relative_position_bias = self.relative_position_bias_table[
            self.relative_position_index.view(-1)
        ].view(N, N, -1)
        relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()
        attn = attn + relative_position_bias.unsqueeze(0)

        if mask is not None:
            # mask: [num_windows, N, N] — used for shifted windows
            nW = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)

        attn = F.softmax(attn, dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


class SwinTransformerBlock(nn.Module):
    def __init__(self, embed_dim, input_resolution, num_heads, window_size=7,
                 shift_size=0, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.input_resolution = input_resolution  # (H, W)
        self.window_size = window_size
        self.shift_size = shift_size

        # If the window doesn't fit (resolution <= window_size), don't shift/partition
        if min(input_resolution) <= window_size:
            self.shift_size = 0
            self.window_size = min(input_resolution)

        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = WindowAttention(embed_dim, self.window_size, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout)
        )

        if self.shift_size > 0:
            self.register_buffer("attn_mask", self._compute_attn_mask())
        else:
            self.attn_mask = None

    def _compute_attn_mask(self):
        H, W = self.input_resolution
        img_mask = torch.zeros((1, H, W, 1))
        h_slices = (slice(0, -self.window_size),
                    slice(-self.window_size, -self.shift_size),
                    slice(-self.shift_size, None))
        w_slices = (slice(0, -self.window_size),
                    slice(-self.window_size, -self.shift_size),
                    slice(-self.shift_size, None))
        cnt = 0
        for h in h_slices:
            for w in w_slices:
                img_mask[:, h, w, :] = cnt
                cnt += 1

        img_mask = img_mask.view(1, H * W, 1)
        mask_windows = window_partition(img_mask, self.window_size, H, W)
        mask_windows = mask_windows.view(-1, self.window_size * self.window_size)
        attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
        attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0))
        attn_mask = attn_mask.masked_fill(attn_mask == 0, float(0.0))
        return attn_mask

    def forward(self, x):
        H, W = self.input_resolution
        B, N, C = x.shape

        shortcut = x
        x = self.norm1(x)
        x = x.view(B, H, W, C)

        if self.shift_size > 0:
            shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
        else:
            shifted_x = x

        shifted_x = shifted_x.view(B, H * W, C)
        x_windows = window_partition(shifted_x, self.window_size, H, W)

        attn_windows = self.attn(x_windows, mask=self.attn_mask)

        shifted_x = window_reverse(attn_windows, self.window_size, H, W)

        if self.shift_size > 0:
            shifted_x = shifted_x.view(B, H, W, C)
            x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
            x = x.view(B, H * W, C)
        else:
            x = shifted_x.view(B, H, W, C).view(B, H * W, C)

        x = shortcut + x
        x = x + self.mlp(self.norm2(x))
        return x


class PatchMerging(nn.Module):
    """Downsamples resolution by 2x and doubles channel dim — the hierarchical part of Swin."""
    def __init__(self, input_resolution, embed_dim):
        super().__init__()
        self.input_resolution = input_resolution
        self.reduction = nn.Linear(4 * embed_dim, 2 * embed_dim, bias=False)
        self.norm = nn.LayerNorm(4 * embed_dim)

    def forward(self, x):
        H, W = self.input_resolution
        B, N, C = x.shape
        x = x.view(B, H, W, C)

        x0 = x[:, 0::2, 0::2, :]
        x1 = x[:, 1::2, 0::2, :]
        x2 = x[:, 0::2, 1::2, :]
        x3 = x[:, 1::2, 1::2, :]
        x = torch.cat([x0, x1, x2, x3], dim=-1)  # [B, H/2, W/2, 4*C]
        x = x.view(B, -1, 4 * C)

        x = self.norm(x)
        x = self.reduction(x)
        return x


class SwinStage(nn.Module):
    def __init__(self, embed_dim, input_resolution, depth, num_heads, window_size,
                 mlp_ratio=4.0, dropout=0.1, downsample=True):
        super().__init__()
        self.blocks = nn.ModuleList([
            SwinTransformerBlock(
                embed_dim, input_resolution, num_heads, window_size,
                shift_size=0 if (i % 2 == 0) else window_size // 2,
                mlp_ratio=mlp_ratio, dropout=dropout
            )
            for i in range(depth)
        ])
        self.downsample = PatchMerging(input_resolution, embed_dim) if downsample else None

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        if self.downsample is not None:
            x = self.downsample(x)
        return x


class SwinTransformer(nn.Module):
    def __init__(self, image_size, patch_size, num_classes, embed_dim,
                 depths, num_heads,
                 window_size=4, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.num_layers = len(depths)
        self.patch_embed = PatchEmbedding(image_size, patch_size, 3, embed_dim)
        self.pos_drop = nn.Dropout(dropout)

        grid_size = image_size // patch_size
        self.stages = nn.ModuleList()
        cur_dim = embed_dim
        cur_res = grid_size

        for i in range(self.num_layers):
            stage = SwinStage(
                embed_dim=cur_dim,
                input_resolution=(cur_res, cur_res),
                depth=depths[i],
                num_heads=num_heads[i],
                window_size=window_size,
                mlp_ratio=mlp_ratio,
                dropout=dropout,
                downsample=(i < self.num_layers - 1)  # last stage doesn't downsample
            )
            self.stages.append(stage)
            if i < self.num_layers - 1:
                cur_dim *= 2
                cur_res //= 2

        self.norm = nn.LayerNorm(cur_dim)
        self.head = nn.Linear(cur_dim, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)
        x = self.pos_drop(x)

        for stage in self.stages:
            x = stage(x)

        x = self.norm(x)
        x = x.mean(dim=1)  # global average pool over remaining tokens (no CLS token in Swin)
        x = self.head(x)
        return x

In [5]:
# Training function
def train():
    model.train()
    for epoch in range(num_epochs):
        start_event = torch.cuda.Event(True)
        end_event = torch.cuda.Event(True)
        start_event.record()
        progress_bar = tqdm(train_loader, desc=f'Epoch [{epoch+1}/{num_epochs}]')
        for i, (images, labels) in enumerate(progress_bar):
            images = images.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(images).logits
            loss = criterion(outputs, labels)

            # Backward and optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if (i+1) % 100 == 0:
                progress_bar.set_postfix({'loss': loss.item()})
        end_event.record()
        torch.cuda.synchronize()
        elapsed_time_ms = start_event.elapsed_time(end_event)
        print(f"Epoch execution time: {(elapsed_time_ms / 1000):.3f} s")

In [6]:
# Testing function
def test():
    model.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        for images, labels in tqdm(test_loader, desc='Testing'):
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images).logits
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total
        print(f'Test Accuracy: {accuracy:.2f}%')

In [7]:
# Run training and testing
def run():
  print("Training started...")
  train()
  print("\nTesting started...")
  test()

  dummy_input = torch.randn(batch_size, 3, image_size, image_size).to(device)

  # Get model summary including FLOPs and parameters
  model_summary = summary(model, input_data=dummy_input, verbose=0)

  print(f"\nTotal theoretical parameters: {model_summary.total_params}")
  print(f"Total FLOPs per forward pass with batch size {batch_size}: {2 * model_summary.total_mult_adds}")

# Problem 2

In [8]:
model = SwinForImageClassification.from_pretrained(
    "microsoft/swin-tiny-patch4-window7-224",
    num_labels = 100,
    ignore_mismatched_sizes = True
  ).to(device)

for p in model.swin.parameters():
  p.requires_grad = False
for p in model.classifier.parameters():
  p.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr = learning_rate)
run()

config.json:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.


model.safetensors:   0%|          | 0.00/113M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/221 [00:00<?, ?it/s]

[transformers] SwinForImageClassification LOAD REPORT from: microsoft/swin-tiny-patch4-window7-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


...

Epoch execution time: 49.343 s


Epoch [5/5]: 100%|██████████| 1563/1563 [00:49<00:00, 31.67it/s, loss=1.67]


Epoch execution time: 49.360 s

Testing started...


Testing: 100%|██████████| 313/313 [00:10<00:00, 29.86it/s]


Test Accuracy: 66.26%

Total theoretical parameters: 27596254
Total FLOPs per forward pass with batch size 32: 3975394708


In [9]:
model = SwinForImageClassification.from_pretrained(
    "microsoft/swin-small-patch4-window7-224",
    num_labels = 100,
    ignore_mismatched_sizes = True
  ).to(device)

for p in model.swin.parameters():
  p.requires_grad = False
for p in model.classifier.parameters():
  p.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr = learning_rate)
run()

config.json:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.


pytorch_model.bin:   0%|          | 0.00/199M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/425 [00:00<?, ?it/s]

[transformers] SwinForImageClassification LOAD REPORT from: microsoft/swin-small-patch4-window7-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


...

Epoch execution time: 83.558 s


Epoch [5/5]: 100%|██████████| 1563/1563 [01:23<00:00, 18.71it/s, loss=1.29]


Epoch execution time: 83.556 s

Testing started...


Testing: 100%|██████████| 313/313 [00:16<00:00, 18.58it/s]


Test Accuracy: 70.35%

Total theoretical parameters: 48914158
Total FLOPs per forward pass with batch size 32: 6701260564


###Scratch

In [10]:
# Hyperparameters
image_size = 32
num_classes = 100
num_epochs = 5
learning_rate = 0.001
batch_size = 32

In [11]:
# Load model directly
from transformers import SwinForImageClassification, AutoImageProcessor, SwinConfig

# Data preparation with proper preprocessing for Swin
processor = AutoImageProcessor.from_pretrained("microsoft/swin-tiny-patch4-window7-224")

transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std)
])


train_data = torchvision.datasets.CIFAR100(
    root='./data', train=True, transform=transform, download=False)
test_data = torchvision.datasets.CIFAR100(
    root='./data', train=False, transform=transform, download=False)

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle = True, num_workers=2, pin_memory=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle = True, num_workers=2, pin_memory=True)


In [12]:
# Training function
def train():
    model.train()
    for epoch in range(num_epochs):
        start_event = torch.cuda.Event(True)
        end_event = torch.cuda.Event(True)
        start_event.record()
        progress_bar = tqdm(train_loader, desc=f'Epoch [{epoch+1}/{num_epochs}]')
        for i, (images, labels) in enumerate(progress_bar):
            images = images.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward and optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if (i+1) % 100 == 0:
                progress_bar.set_postfix({'loss': loss.item()})
        end_event.record()
        torch.cuda.synchronize()
        elapsed_time_ms = start_event.elapsed_time(end_event)
        print(f"Epoch execution time: {(elapsed_time_ms / 1000):.3f} s")

In [13]:
# Testing function
def test():
    model.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        for images, labels in tqdm(test_loader, desc='Testing'):
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total
        print(f'Test Accuracy: {accuracy:.2f}%')

In [14]:
model = SwinTransformer(
    image_size=32,
    patch_size = 4,
    num_classes = num_classes,
    embed_dim = 256,
    depths = (4,),
    num_heads=(4,),
    window_size = 4,
    mlp_ratio = 4,
    dropout = 0.1
  ).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = learning_rate)
run()

Training started...


Epoch [1/5]: 100%|██████████| 1563/1563 [00:23<00:00, 66.78it/s, loss=4.18]


Epoch execution time: 23.407 s


Epoch [2/5]: 100%|██████████| 1563/1563 [00:23<00:00, 65.90it/s, loss=3.81]


Epoch execution time: 23.721 s


...

Epoch execution time: 23.166 s


Epoch [5/5]: 100%|██████████| 1563/1563 [00:23<00:00, 65.64it/s, loss=3.96]


Epoch execution time: 23.814 s

Testing started...


Testing: 100%|██████████| 313/313 [00:01<00:00, 168.57it/s]

Test Accuracy: 12.88%

Total theoretical parameters: 3199092
Total FLOPs per forward pass with batch size 32: 457382144
